# Phase 3 — Feature Engineering & Representation Foundation
## AI-Powered Fake Job and Recruitment Scam Detection System

**Objective**: Demonstrate modular feature extraction, named feature groups, multi-representation interfaces, dimension verification, and sparsity analysis without training final classification models.

In [ ]:
import os
import sys
import json
import pandas as pd
import numpy as np
from scipy import sparse

# Ensure project root is in python path
project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.features.builder import ModularFeatureBuilder
from src.features.feature_catalog import FEATURE_CATALOG_REGISTRY
print("Project imports loaded successfully.")

### 1. Load Processed Datasets (Established Train/Test Split)
We load the verified 80/20 train/test split created in Phase 2 ($N_{\text{train}}=14,304, N_{\text{test}}=3,576$).

In [ ]:
df_train = pd.read_csv('../data/processed/train.csv')
df_test = pd.read_csv('../data/processed/test.csv')

print(f"Training set shape: {df_train.shape}")
print(f"Test set shape:     {df_test.shape}")
print(f"Training Fraud Rate: {df_train['fraudulent'].mean():.4%}")
print(f"Test Fraud Rate:     {df_test['fraudulent'].mean():.4%}")

### 2. Fit Modular Feature Builder Strictly on Training Data
All vocabulary, inverse document frequencies, categorical encoders, and scalers are fitted exclusively on `df_train`.

In [ ]:
builder = ModularFeatureBuilder(
    tfidf_max_features=10000,
    tfidf_ngram_range=(1, 2),
    heuristic_config_path='../config/heuristic_keywords.yaml'
)

builder.fit(df_train)
print("ModularFeatureBuilder fitted successfully on training partition.")

### 3. Extract and Inspect Named Feature Groups
We evaluate each feature group individually to verify dimensional consistency and sparsity across training and test partitions.

In [ ]:
summary = builder.get_dimensions_summary(df_train, df_test)

df_summary = pd.DataFrame([
    {
        "Feature Group": group,
        "Num Features": data["num_features"],
        "Train Shape": str(data["train_shape"]),
        "Test Shape": str(data["test_shape"]),
        "Format": "Sparse (CSR)" if data["is_sparse"] else "Dense (NumPy)",
        "Sparsity (%)": f"{data['sparsity_percent']:.2f}%"
    }
    for group, data in summary.items()
])

print(df_summary.to_string(index=False))

### 4. Sample Feature Inspection: Suspicious Language Heuristics
Let's inspect how the heuristic features identify scam language patterns in fraudulent vs legitimate postings.

In [ ]:
df_heur_train = builder.heuristic_extractor.transform(df_train)
df_heur_train['fraudulent'] = df_train['fraudulent'].values

heur_comparison = df_heur_train.groupby('fraudulent')[
    ['payment_requests_presence', 'urgency_presence', 'guaranteed_income_presence',
     'work_from_home_claims_presence', 'personal_info_requests_presence', 'has_any_scam_keyword']
].mean() * 100

print("Percentage of Postings Triggering Heuristic Categories (%):")
print(heur_comparison.round(2).to_string())

### 5. Inspect Text Statistics & Structural Ratios
Comparing document length and structural completeness between legitimate and fraudulent postings.

In [ ]:
df_stats_train = builder.text_stats_extractor.transform(df_train)
df_stats_train['fraudulent'] = df_train['fraudulent'].values

stats_comparison = df_stats_train.groupby('fraudulent')[
    ['company_profile_char_count', 'description_char_count', 'populated_sections_count',
     'composite_uppercase_ratio', 'desc_to_req_length_ratio']
].median()

print("Median Text Statistics by Class:")
print(stats_comparison.round(3).to_string())

### 6. Full Hybrid Feature Matrix Generation (`HYBRID_ALL`)
Generate and verify the complete sparse CSR hybrid matrix combining TF-IDF and all tabular/heuristic features.

In [ ]:
X_train_hybrid = builder.transform(df_train)
X_test_hybrid = builder.transform(df_test)

print(f"X_train Hybrid Matrix Shape: {X_train_hybrid.shape}")
print(f"X_test Hybrid Matrix Shape:  {X_test_hybrid.shape}")
print(f"X_train Matrix Type:         {type(X_train_hybrid)}")
print(f"X_train Memory Usage:        {X_train_hybrid.data.nbytes / (1024**2):.2f} MB")

### 7. Feature Names & Documentation Registry Verification

In [ ]:
all_feature_names = builder.get_feature_names()
print(f"Total Unique Feature Names: {len(all_feature_names)}")
print("Sample Feature Names:")
print(all_feature_names[:10])
print(all_feature_names[10000:10015])

# Ensure no target leakage
assert 'fraudulent' not in all_feature_names, "Target leakage detected!"
assert 'job_id' not in all_feature_names, "ID leakage detected!"
print("Zero leakage checks passed successfully.")